# Teste alinhado — COBYLA tradicional vs Transformer one-shot

Este notebook foi preparado para ser **colado ao final do notebook em que o problema VQE já foi construído**.

Ele compara os dois métodos usando:

- o mesmo `stocks_prices.csv`;
- os mesmos retornos e a mesma matriz de covariância;
- o mesmo Hamiltoniano `ising`;
- o mesmo ansatz de Dicke;
- a mesma função de energia `exp_val`;
- uma execução normal do COBYLA;
- uma inferência do Transformer;
- a mesma avaliação física final para os dois vetores.

## Quantidade de interações

Para o COBYLA, a medida mais justa é o número de chamadas à função objetivo:

\[
N_{\mathrm{oracle}}^{\mathrm{COBYLA}}
=
\text{número de avaliações de }E(\theta).
\]

Para o Transformer, durante a inferência:

\[
N_{\mathrm{forward}}^{\mathrm{Transformer}}=1.
\]

Depois, fazemos uma avaliação física de sua saída:

\[
N_{\mathrm{oracle}}^{\mathrm{Transformer}}=1.
\]

O Transformer não possui uma trajetória iterativa durante a inferência: ele produz os 30 parâmetros diretamente.

In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO
# ============================================================

from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from qiskit.quantum_info import Statevector


# Procura o mesmo stocks_prices.csv usado no projeto.
CSV_CANDIDATES = [
    Path("stocks_prices.csv"),
    Path("../stocks_prices.csv"),
    Path("data/stocks_prices.csv"),
    Path("../data/stocks_prices.csv"),
    Path("data/raw/stocks_prices.csv"),
    Path("../data/raw/stocks_prices.csv"),
]

STOCKS_CSV = next(
    (path.resolve() for path in CSV_CANDIDATES if path.exists()),
    None,
)

if STOCKS_CSV is None:
    raise FileNotFoundError(
        "stocks_prices.csv não foi encontrado. "
        "Defina manualmente a variável STOCKS_CSV."
    )

# Escolha automática do modelo.
# Caso existam os dois, prioriza o modelo treinado sem a região boa de theta_17.
MODEL_TO_TEST = globals().get(
    "model_theta17_fora",
    globals().get("model", None),
)

MODEL_LABEL = (
    "Transformer sem região boa de theta_17"
    if "model_theta17_fora" in globals()
    else "Transformer original"
)

TOP_K_BITSTRINGS = 12
WRAP_TRANSFORMER_OUTPUT = True
SAVE_RESULTS = True

OUTPUT_DIR = Path("results/comparacao_cobyla_transformer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CSV:", STOCKS_CSV)
print("Modelo:", MODEL_LABEL)
print("Saída:", OUTPUT_DIR.resolve())

In [ ]:
# ============================================================
# 2. VERIFICAÇÕES DE COMPATIBILIDADE
# ============================================================

required_names = [
    "ising",
    "ansatz",
    "exp_val",
    "optimizers_dict",
    "initial_points_dict",
    "device",
]

missing = [
    name for name in required_names
    if name not in globals()
]

if MODEL_TO_TEST is None:
    missing.append("model ou model_theta17_fora")

if missing:
    raise RuntimeError(
        "Execute primeiro as células de construção do projeto. "
        f"Objetos ausentes: {missing}"
    )

if "COBYLA" not in optimizers_dict:
    raise KeyError("optimizers_dict não contém a chave 'COBYLA'.")

N_PARAMS = int(ansatz.num_parameters)

COBYLA_INITIAL_POINT = np.asarray(
    initial_points_dict["Dicke_state"][0],
    dtype=float,
).reshape(-1)

if COBYLA_INITIAL_POINT.size != N_PARAMS:
    raise ValueError(
        f"O ansatz possui {N_PARAMS} parâmetros, mas o ponto inicial "
        f"possui {COBYLA_INITIAL_POINT.size}."
    )

print("Pré-teste aprovado.")
print("Número de qubits:", ising.num_qubits)
print("Número de parâmetros:", N_PARAMS)
print("Dispositivo:", device)

In [ ]:
# ============================================================
# 3. CONSTRUIR A MESMA ENTRADA A PARTIR DO stocks_prices.csv
# ============================================================

prices_raw = pd.read_csv(STOCKS_CSV)
first_column = prices_raw.columns[0]

# Se a primeira coluna for temporal, usa como índice e a remove das features.
if not pd.api.types.is_numeric_dtype(prices_raw[first_column]):
    parsed_dates = pd.to_datetime(
        prices_raw[first_column],
        errors="coerce",
    )

    if parsed_dates.notna().mean() > 0.8:
        prices_raw = prices_raw.drop(columns=[first_column])
        prices_raw.index = parsed_dates

prices = prices_raw.select_dtypes(include=[np.number]).copy()
prices = prices.dropna(axis=1, how="all")
prices = prices.replace([np.inf, -np.inf], np.nan)
prices = prices.ffill().bfill().dropna()

if prices.shape[1] != ising.num_qubits:
    raise ValueError(
        "O CSV possui quantidade de ativos diferente da quantidade de qubits. "
        f"CSV={prices.shape[1]}, qubits={ising.num_qubits}. "
        "Confirme se este é exatamente o arquivo usado para construir o Hamiltoniano."
    )

returns = prices.pct_change().dropna()

assets_return_compare = returns.mean(axis=0).to_numpy(dtype=np.float32)
covariance_compare = returns.cov().to_numpy(dtype=np.float32)

assets_return_tensor = torch.tensor(
    assets_return_compare,
    dtype=torch.float32,
)

covariance_tensor = torch.tensor(
    covariance_compare,
    dtype=torch.float32,
)

# Formato por ativo: [retorno médio | linha da matriz de covariância]
X_compare = torch.cat(
    [
        assets_return_tensor.unsqueeze(-1),
        covariance_tensor,
    ],
    dim=1,
)

entrada_compare = X_compare.unsqueeze(0)

print("Ativos:", list(prices.columns))
print("Formato dos preços:", prices.shape)
print("Formato dos retornos:", returns.shape)
print("Entrada do Transformer:", entrada_compare.shape)

assert entrada_compare.shape == (
    1,
    ising.num_qubits,
    ising.num_qubits + 1,
)

In [ ]:
# ============================================================
# 4. FUNÇÕES AUXILIARES
# ============================================================

def scalar_energy(value):
    """Converte a saída de exp_val em float."""
    return float(np.asarray(value).reshape(-1)[0])


def wrap_angles(theta):
    """Coloca todos os ângulos em [-pi, pi)."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    return (theta + np.pi) % (2 * np.pi) - np.pi


def bind_ansatz(theta):
    """Liga explicitamente o vetor theta ao ansatz."""
    theta = np.asarray(theta, dtype=float).reshape(-1)

    if theta.size != ansatz.num_parameters:
        raise ValueError(
            f"Esperados {ansatz.num_parameters} parâmetros, "
            f"recebidos {theta.size}."
        )

    parameter_order = list(ansatz.parameters)
    parameter_map = {
        parameter: float(theta[index])
        for index, parameter in enumerate(parameter_order)
    }

    return ansatz.assign_parameters(parameter_map, inplace=False)


def evaluate_statevector(theta):
    """
    Avalia o vetor no mesmo problema:
    - energia pela mesma exp_val;
    - distribuição completa de probabilidades;
    - bitstring dominante;
    - probabilidade dominante.
    """
    theta = np.asarray(theta, dtype=float).reshape(-1)
    energy = scalar_energy(exp_val(theta))

    bound = bind_ansatz(theta)
    bound = bound.remove_final_measurements(inplace=False)
    state = Statevector.from_instruction(bound)

    probability_dict = {
        str(bitstring): float(probability)
        for bitstring, probability in state.probabilities_dict().items()
    }

    dominant_bitstring = max(probability_dict, key=probability_dict.get)

    return {
        "energy": energy,
        "probabilities": probability_dict,
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": probability_dict[dominant_bitstring],
    }


def infer_exact_reference():
    """Tenta localizar energia e bitstring exatos já existentes no notebook."""
    exact_energy = globals().get("EXACT_ENERGY", None)
    exact_bitstring = globals().get("EXACT_BITSTRING", None)

    if exact_energy is None and "exact_result" in globals():
        try:
            offset_value = globals().get("offset", 0.0)
            exact_energy = float(
                np.real(exact_result.eigenvalue + offset_value)
            )
        except Exception:
            exact_energy = None

    if exact_bitstring is None and "exact_result" in globals():
        try:
            exact_state = exact_result.eigenstate.to_dict()
            exact_bitstring = max(
                exact_state,
                key=lambda key: abs(exact_state[key]) ** 2,
            )
        except Exception:
            exact_bitstring = None

    if exact_bitstring is None:
        for candidate in ["best_answer", "optimal_bitstring"]:
            if candidate in globals():
                exact_bitstring = str(globals()[candidate])
                break

    if exact_energy is not None:
        exact_energy = float(exact_energy)

    if exact_bitstring is not None:
        exact_bitstring = str(exact_bitstring).strip()

    return exact_energy, exact_bitstring


EXACT_ENERGY_COMPARE, EXACT_BITSTRING_COMPARE = infer_exact_reference()

print("Energia exata localizada:", EXACT_ENERGY_COMPARE)
print("Bitstring exato localizado:", EXACT_BITSTRING_COMPARE)

In [ ]:
# ============================================================
# 5. COBYLA TRADICIONAL COM RASTREAMENTO
# ============================================================

cobyla_energy_history = []
cobyla_theta_history = []


def exp_val_tracked(theta):
    """
    Cada chamada desta função corresponde a uma interação do COBYLA
    com a função objetivo/oráculo de energia.
    """
    theta = np.asarray(theta, dtype=float).reshape(-1)
    value = scalar_energy(exp_val(theta))

    cobyla_energy_history.append(value)
    cobyla_theta_history.append(theta.copy())

    return value


cobyla_start = perf_counter()

result_cobyla = optimizers_dict["COBYLA"].minimize(
    fun=exp_val_tracked,
    x0=COBYLA_INITIAL_POINT.copy(),
)

cobyla_optimization_time = perf_counter() - cobyla_start

theta_cobyla = np.asarray(result_cobyla.x, dtype=float).reshape(-1)

# Validação final comum aos dois métodos.
cobyla_validation_start = perf_counter()
evaluation_cobyla = evaluate_statevector(theta_cobyla)
cobyla_validation_time = perf_counter() - cobyla_validation_start

cobyla_nfev = int(
    getattr(result_cobyla, "nfev", len(cobyla_energy_history))
)

cobyla_nit = getattr(result_cobyla, "nit", None)

print("=" * 70)
print("COBYLA FINALIZADO")
print("=" * 70)
print("Avaliações da função objetivo:", cobyla_nfev)
print("Iterações reportadas:", cobyla_nit)
print("Tempo de otimização:", cobyla_optimization_time)
print("Energia final:", evaluation_cobyla["energy"])
print("Bitstring dominante:", evaluation_cobyla["dominant_bitstring"])
print("Probabilidade dominante:", evaluation_cobyla["dominant_probability"])

In [ ]:
# ============================================================
# 6. TRANSFORMER ONE-SHOT
# ============================================================

MODEL_TO_TEST = MODEL_TO_TEST.to(device)
MODEL_TO_TEST.eval()

entrada_modelo = entrada_compare.to(
    device=device,
    dtype=torch.float32,
)

transformer_start = perf_counter()

with torch.no_grad():
    transformer_output = MODEL_TO_TEST(entrada_modelo)

transformer_inference_time = perf_counter() - transformer_start

theta_transformer = (
    transformer_output
    .detach()
    .cpu()
    .reshape(-1)
    .numpy()
    .astype(float)
)

if theta_transformer.size != N_PARAMS:
    raise ValueError(
        f"O Transformer produziu {theta_transformer.size} valores, "
        f"mas o ansatz exige {N_PARAMS}."
    )

if WRAP_TRANSFORMER_OUTPUT:
    theta_transformer = wrap_angles(theta_transformer)

# Uma única avaliação física da saída prevista.
transformer_validation_start = perf_counter()
evaluation_transformer = evaluate_statevector(theta_transformer)
transformer_validation_time = perf_counter() - transformer_validation_start

transformer_forward_passes = 1
transformer_oracle_evaluations = 1

print("=" * 70)
print("TRANSFORMER FINALIZADO")
print("=" * 70)
print("Forward passes:", transformer_forward_passes)
print("Avaliações físicas para validação:", transformer_oracle_evaluations)
print("Tempo de inferência:", transformer_inference_time)
print("Energia final:", evaluation_transformer["energy"])
print("Bitstring dominante:", evaluation_transformer["dominant_bitstring"])
print("Probabilidade dominante:", evaluation_transformer["dominant_probability"])
print("theta_17 previsto:", theta_transformer[17])

In [ ]:
# ============================================================
# 7. TABELA ALINHADA DOS RESULTADOS
# ============================================================

def probability_of_exact(evaluation):
    if EXACT_BITSTRING_COMPARE is None:
        return np.nan

    return float(
        evaluation["probabilities"].get(EXACT_BITSTRING_COMPARE, 0.0)
    )


def energy_gap(energy):
    if EXACT_ENERGY_COMPARE is None:
        return np.nan

    return abs(float(energy) - EXACT_ENERGY_COMPARE)


comparison_df = pd.DataFrame(
    [
        {
            "metodo": "COBYLA",
            "energia": evaluation_cobyla["energy"],
            "gap_energia": energy_gap(evaluation_cobyla["energy"]),
            "bitstring_dominante": evaluation_cobyla["dominant_bitstring"],
            "probabilidade_dominante": evaluation_cobyla["dominant_probability"],
            "probabilidade_bitstring_exato": probability_of_exact(evaluation_cobyla),
            "acertou_bitstring_exato": (
                evaluation_cobyla["dominant_bitstring"] == EXACT_BITSTRING_COMPARE
                if EXACT_BITSTRING_COMPARE is not None
                else np.nan
            ),
            "avaliacoes_objetivo_oraculo": cobyla_nfev,
            "forward_passes": 0,
            "tempo_geracao_s": cobyla_optimization_time,
            "tempo_validacao_s": cobyla_validation_time,
            "tempo_total_s": cobyla_optimization_time + cobyla_validation_time,
        },
        {
            "metodo": MODEL_LABEL,
            "energia": evaluation_transformer["energy"],
            "gap_energia": energy_gap(evaluation_transformer["energy"]),
            "bitstring_dominante": evaluation_transformer["dominant_bitstring"],
            "probabilidade_dominante": evaluation_transformer["dominant_probability"],
            "probabilidade_bitstring_exato": probability_of_exact(evaluation_transformer),
            "acertou_bitstring_exato": (
                evaluation_transformer["dominant_bitstring"] == EXACT_BITSTRING_COMPARE
                if EXACT_BITSTRING_COMPARE is not None
                else np.nan
            ),
            "avaliacoes_objetivo_oraculo": transformer_oracle_evaluations,
            "forward_passes": transformer_forward_passes,
            "tempo_geracao_s": transformer_inference_time,
            "tempo_validacao_s": transformer_validation_time,
            "tempo_total_s": transformer_inference_time + transformer_validation_time,
        },
    ]
)

display(comparison_df.T)

if SAVE_RESULTS:
    comparison_df.to_csv(
        OUTPUT_DIR / "comparacao_resumo.csv",
        index=False,
    )

In [ ]:
# ============================================================
# 8. GRÁFICO — CONVERGÊNCIA DA ENERGIA
# ============================================================

history = np.asarray(cobyla_energy_history, dtype=float)
x_cobyla = np.arange(1, len(history) + 1)

fig, ax = plt.subplots(figsize=(11, 5.5))

ax.plot(
    x_cobyla,
    history,
    linewidth=1.5,
    label="COBYLA — energia em cada avaliação",
)

ax.axhline(
    evaluation_transformer["energy"],
    linestyle="--",
    linewidth=2,
    label=f"{MODEL_LABEL} — previsão one-shot",
)

if EXACT_ENERGY_COMPARE is not None:
    ax.axhline(
        EXACT_ENERGY_COMPARE,
        linestyle=":",
        linewidth=2,
        label="Energia exata",
    )

ax.scatter(
    [len(history)],
    [evaluation_cobyla["energy"]],
    s=70,
    zorder=5,
    label="COBYLA final",
)

ax.set_title(
    "Projeção da energia: trajetória do COBYLA vs previsão do Transformer"
)
ax.set_xlabel("Número acumulado de avaliações da função objetivo")
ax.set_ylabel("Energia")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

if SAVE_RESULTS:
    fig.savefig(
        OUTPUT_DIR / "01_convergencia_energia.png",
        dpi=250,
        bbox_inches="tight",
    )

plt.show()

In [ ]:
# ============================================================
# 9. GRÁFICO — QUANTIDADE DE INTERAÇÕES
# ============================================================

interaction_plot = pd.DataFrame(
    {
        "metodo": ["COBYLA", MODEL_LABEL],
        "avaliacoes_fisicas": [
            cobyla_nfev,
            transformer_oracle_evaluations,
        ],
    }
)

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    interaction_plot["metodo"],
    interaction_plot["avaliacoes_fisicas"],
)

for bar, value in zip(bars, interaction_plot["avaliacoes_fisicas"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(value)}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

ax.set_title("Interações com a função objetivo/circuito")
ax.set_ylabel("Número de avaliações físicas")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

if SAVE_RESULTS:
    fig.savefig(
        OUTPUT_DIR / "02_interacoes_oraculo.png",
        dpi=250,
        bbox_inches="tight",
    )

plt.show()

print(
    "Redução aproximada de avaliações:",
    cobyla_nfev / max(transformer_oracle_evaluations, 1),
    "vezes",
)

In [ ]:
# ============================================================
# 10. GRÁFICO — MÉTRICAS FINAIS ALINHADAS
# ============================================================

metrics_for_plot = comparison_df.set_index("metodo")[
    [
        "gap_energia",
        "probabilidade_bitstring_exato",
        "tempo_total_s",
    ]
].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

metric_specs = [
    ("gap_energia", "Gap energético", "Menor é melhor"),
    (
        "probabilidade_bitstring_exato",
        "P(bitstring exato)",
        "Maior é melhor",
    ),
    ("tempo_total_s", "Tempo total (s)", "Menor é melhor"),
]

for ax, (column, title, subtitle) in zip(axes, metric_specs):
    values = metrics_for_plot[column]
    bars = ax.bar(values.index, values.values)

    ax.set_title(f"{title}\n{subtitle}")
    ax.tick_params(axis="x", rotation=18)
    ax.grid(axis="y", alpha=0.25)

    for bar, value in zip(bars, values.values):
        if np.isfinite(value):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{value:.6f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

fig.suptitle("Comparação final alinhada — mesmo problema e mesmo ansatz")
fig.tight_layout()

if SAVE_RESULTS:
    fig.savefig(
        OUTPUT_DIR / "03_metricas_finais.png",
        dpi=250,
        bbox_inches="tight",
    )

plt.show()

In [ ]:
# ============================================================
# 11. GRÁFICO — DISTRIBUIÇÃO DOS BITSTRINGS
# ============================================================

all_bitstrings = set(evaluation_cobyla["probabilities"]) | set(
    evaluation_transformer["probabilities"]
)

ranked_bitstrings = sorted(
    all_bitstrings,
    key=lambda bitstring: max(
        evaluation_cobyla["probabilities"].get(bitstring, 0.0),
        evaluation_transformer["probabilities"].get(bitstring, 0.0),
    ),
    reverse=True,
)[:TOP_K_BITSTRINGS]

bitstring_df = pd.DataFrame(
    {
        "bitstring": ranked_bitstrings,
        "COBYLA": [
            evaluation_cobyla["probabilities"].get(bitstring, 0.0)
            for bitstring in ranked_bitstrings
        ],
        MODEL_LABEL: [
            evaluation_transformer["probabilities"].get(bitstring, 0.0)
            for bitstring in ranked_bitstrings
        ],
    }
)

x = np.arange(len(bitstring_df))
width = 0.38

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(
    x - width / 2,
    bitstring_df["COBYLA"],
    width,
    label="COBYLA",
)

ax.bar(
    x + width / 2,
    bitstring_df[MODEL_LABEL],
    width,
    label=MODEL_LABEL,
)

ax.set_xticks(x)
ax.set_xticklabels(
    bitstring_df["bitstring"],
    rotation=45,
    ha="right",
)
ax.set_ylabel("Probabilidade")
ax.set_title("Top bitstrings — distribuições dos dois métodos")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()

if SAVE_RESULTS:
    fig.savefig(
        OUTPUT_DIR / "04_distribuicao_bitstrings.png",
        dpi=250,
        bbox_inches="tight",
    )

plt.show()
display(bitstring_df)

In [ ]:
# ============================================================
# 12. GRÁFICO — COMPARAÇÃO DOS 30 THETA
# ============================================================

theta_difference = wrap_angles(theta_transformer - theta_cobyla)
theta_indices = np.arange(N_PARAMS)

fig, ax = plt.subplots(figsize=(14, 5.5))

ax.plot(
    theta_indices,
    wrap_angles(theta_cobyla),
    marker="o",
    linewidth=1.4,
    label="COBYLA",
)

ax.plot(
    theta_indices,
    wrap_angles(theta_transformer),
    marker="s",
    linewidth=1.4,
    label=MODEL_LABEL,
)

ax.set_xticks(theta_indices)
ax.set_xlabel("Índice do parâmetro")
ax.set_ylabel("Ângulo em [-pi, pi)")
ax.set_title("Vetores de parâmetros produzidos pelos dois métodos")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

if SAVE_RESULTS:
    fig.savefig(
        OUTPUT_DIR / "05_comparacao_theta.png",
        dpi=250,
        bbox_inches="tight",
    )

plt.show()

theta_comparison_df = pd.DataFrame(
    {
        "theta_index": theta_indices,
        "theta_name": [
            f"theta_{index:02d}" for index in theta_indices
        ],
        "theta_cobyla": wrap_angles(theta_cobyla),
        "theta_transformer": wrap_angles(theta_transformer),
        "diferenca_circular": theta_difference,
        "diferenca_circular_abs": np.abs(theta_difference),
    }
)

display(
    theta_comparison_df.sort_values(
        "diferenca_circular_abs",
        ascending=False,
    ).head(10)
)

if SAVE_RESULTS:
    theta_comparison_df.to_csv(
        OUTPUT_DIR / "comparacao_theta.csv",
        index=False,
    )

In [ ]:
# ============================================================
# 13. CONCLUSÃO AUTOMÁTICA
# ============================================================

cobyla_gap = energy_gap(evaluation_cobyla["energy"])
transformer_gap = energy_gap(evaluation_transformer["energy"])

cobyla_exact_probability = probability_of_exact(evaluation_cobyla)
transformer_exact_probability = probability_of_exact(evaluation_transformer)

print("=" * 76)
print("CONCLUSÃO ALINHADA")
print("=" * 76)

print(f"COBYLA usou {cobyla_nfev} avaliações da função objetivo.")
print(
    "O Transformer usou 1 forward pass e 1 avaliação física "
    "para validar sua saída."
)
print()

if np.isfinite(cobyla_gap) and np.isfinite(transformer_gap):
    if transformer_gap <= cobyla_gap:
        print(
            "Energia: o Transformer igualou ou superou o COBYLA "
            "nesta execução."
        )
    else:
        print("Energia: o COBYLA produziu menor gap nesta execução.")
else:
    print(
        "Energia exata não foi localizada; compare as energias "
        "diretamente na tabela."
    )

if (
    np.isfinite(cobyla_exact_probability)
    and np.isfinite(transformer_exact_probability)
):
    if transformer_exact_probability >= cobyla_exact_probability:
        print(
            "Bitstring: o Transformer concentrou probabilidade igual "
            "ou maior no bitstring ótimo."
        )
    else:
        print(
            "Bitstring: o COBYLA concentrou maior probabilidade "
            "no bitstring ótimo."
        )

print()
print(
    "Observação: uma execução é um estudo de caso. Para uma comparação "
    "estatística, repita com várias sementes/pontos iniciais do COBYLA."
)

if "model_theta17_fora" in globals():
    theta17_value = float(theta_transformer[17])
    theta17_outside = not (1.18 <= theta17_value <= 1.57)

    print()
    print(f"theta_17 do Transformer restrito: {theta17_value:.6f}")
    print("Fora da região removida:", theta17_outside)

    if (
        theta17_outside
        and np.isfinite(transformer_gap)
        and transformer_gap < 1e-3
    ):
        print(
            "Leitura exploratória: o modelo permaneceu fora da região "
            "removida e ainda obteve energia quase ótima, sustentando "
            "compensação paramétrica local."
        )

## Como interpretar

### COBYLA

- `avaliacoes_objetivo_oraculo` conta quantas vezes o método consultou \(E(\theta)\);
- a curva mostra sua trajetória completa;
- `nfev` é mais adequado que `nit` para comparar custo físico, porque uma iteração pode conter mais de uma avaliação.

### Transformer

- realiza um único forward pass;
- produz os 30 parâmetros diretamente;
- sua saída é avaliada uma vez no mesmo circuito;
- não existe trajetória iterativa na inferência.

### Evidência favorável ao Transformer

A comparação é forte quando ele:

1. usa uma única previsão;
2. produz gap energético próximo ao COBYLA;
3. preserva o bitstring ótimo;
4. mantém probabilidade alta no estado ótimo;
5. reduz fortemente o número de avaliações físicas.

Para o modelo sem a região boa de `theta_17`, permanecer fora da região removida e preservar o desempenho sustenta compensação paramétrica local. Perder o desempenho sustenta importância estrutural local dessa região.